In [ ]:
from astropy.io import ascii

import os
import sys
import glob
import numpy as np
from tqdm import trange
from astropy.io import fits
from astropy.table import Table, vstack
from astropy.convolution import convolve, Gaussian1DKernel
import astropy.units as u
import astropy.coordinates as coord
import matplotlib
import matplotlib.pyplot as plt
from astropy.table import Column
from tqdm import trange
import pandas as pd
import fitsio
from astropy.table import Table, vstack
from astropy import units as u
from astropy.coordinates import SkyCoord
from easyquery import Query, QueryMaker
from scipy.stats import binomtest
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm
from matplotlib.colors import ListedColormap, BoundaryNorm
import h5py
from astropy.cosmology import Planck18

mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['axes.linewidth'] = 1.5
mpl.rcParams['axes.xmargin'] = 1
mpl.rcParams['xtick.labelsize'] = 'x-large'
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.major.width'] = 1.5
mpl.rcParams['ytick.labelsize'] = 'x-large'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.major.width'] = 1.5
mpl.rcParams['legend.frameon'] = False

rootdir = '/global/u1/v/virajvm/'
sys.path.append(os.path.join(rootdir, 'DESI2_LOWZ/desi_dwarfs/code'))

from desi_lowz_funcs import make_subplots, match_c_to_catalog, print_radecs
from desi_lowz_funcs import calc_normalized_dist
from desi_lowz_funcs import find_objects_nearby
# from construct_dwarf_galaxy_catalogs import process_sga_matches
from catalog_paper_plots import make_bar_pie


%load_ext autoreload
%autoreload 2

    

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

clean_cat = Table.read("/pscratch/sd/v/virajvm/catalog_dr1_dwarfs/desi_y1_dwarf_clean_catalog_v4.fits")
# clean_cat = clean_cat[:50000]

In [ ]:
##match the clean cat with other catalogs
gswlc_cat = ascii.read("/pscratch/sd/v/virajvm/desi2_lowz_data/catalogs/GSWLC-X2.dat")
iron = SkyCoord(np.array(clean_cat["RA"])*u.degree, np.array(clean_cat["DEC"])*u.degree  )
gswlc = SkyCoord(np.array(gswlc_cat["RA"])*u.degree, np.array(gswlc_cat["DEC"])*u.degree  )
idx, d2d, _ = iron.match_to_catalog_sky(gswlc)
clean_cat_gswlc_match = clean_cat[d2d.arcsec < 1]
gswlc_match = gswlc_cat[idx][d2d.arcsec < 1]

##these are stellar masses from Hu Zhou XMPG paper. They also use CIGALE here and no AGN is used
hu_cat= Table.read("/global/cfs/cdirs/desi/users/dscholte/data_to_share/sample_catalog_viraj_29052024.fits")
iron = SkyCoord(np.array(clean_cat["RA"])*u.degree, np.array(clean_cat["DEC"])*u.degree  )
hu = SkyCoord(np.array(hu_cat["RA"])*u.degree, np.array(hu_cat["DEC"])*u.degree  )
idx, d2d, _ = iron.match_to_catalog_sky(hu)
clean_cat_hu_match = clean_cat[d2d.arcsec < 1]
hu_match = hu_cat[idx][d2d.arcsec < 1]

###FASTSPECFIT
print("Reading fastspecfit!")
iron_vac = fits.open("/global/cfs/cdirs/desi/public/dr1/vac/dr1/fastspecfit/iron/v2.1/catalogs/fastspec-iron.fits")
fspec_mstar = iron_vac[1].data["LOGMSTAR"]
fspec_ra = iron_vac[2].data["RA"]
fspec_dec = iron_vac[2].data["DEC"]
catalog = SkyCoord(ra= fspec_ra* u.degree, dec= fspec_dec*u.degree )
c = SkyCoord(ra=np.array(clean_cat["RA"])*u.degree, dec=np.array(clean_cat["DEC"])*u.degree )
idx, d2d, d3d = c.match_to_catalog_sky(catalog)
fspec_mstar_f = fspec_mstar[idx][d2d.arcsec < 1]
clean_cat_fspec_match = clean_cat[d2d.arcsec < 1]
print("Finished matching fastspecfit!")

In [ ]:
import cmasher as cmr
from scipy.stats import median_abs_deviation

def measure_bias_scatter(quant_1, quant_2):
    '''
    Meausure the median of quant_1 - quant_2 and the scatter in this difference. We restrict ourselves to objects
    '''

    quant_1f = quant_1[~np.isnan(quant_1) & ~np.isnan(quant_2) ]
    quant_2f = quant_2[~np.isnan(quant_2) & ~np.isnan(quant_2) ]

    med_val = np.median(quant_1f - quant_2f)
    scatters = quant_1f - quant_2f - med_val

    sigma =  median_abs_deviation(scatters, scale='normal')

    print(med_val, sigma)
    return med_val, sigma


In [ ]:
from desi_lowz_funcs import make_alternating_plot

In [ ]:
ax = make_subplots(ncol = 4,nrow = 1,col_spacing = 0.25)

title_size = 15

xmstar = "LOGM_M24_VCMB"

cmap = cmr.gothic_r.copy()
cmap.set_under(alpha=0.)

####
xpos = 7.1
ypos = 6.25
fsize = 14

ax_id = 0
ax[ax_id].set_title(r"CIGALE (no AGN)",fontsize = title_size )
h, xedges, yedges, im=ax[ax_id].hist2d(clean_cat_hu_match[xmstar],hu_match["LOGMSTAR_HU"],range= ( (6,9.5),(6,9.5)),bins= 50,norm=LogNorm(vmin=1,vmax=1000) ,cmap=cmap, rasterized=True)

bias, scatter = measure_bias_scatter(clean_cat_hu_match[xmstar].data,hu_match["LOGMSTAR_HU"])

ax[ax_id].text( xpos,ypos,rf"b = {bias:.2f}, $\sigma$ = {scatter:.2f}",fontsize = fsize)


 # Create a colorbar
cbar = plt.colorbar(im, ax=ax[ax_id], orientation='horizontal', pad=0.05)
cbar.ax.set_position([
    0.04,   # Left position
    0.62,  # Top position
    ax[ax_id].get_position().width * 0.1,  # Width (40% of plot width)
    0.02  # Height (thin bar)
])

###

ax_id = 1
ax[ax_id].set_title(r"GSWLC",fontsize = title_size )
h, xedges, yedges, im=  ax[ax_id].hist2d(clean_cat_gswlc_match[xmstar],gswlc_match["LOGMSTAR"],range= ( (6,9.5),(6,9.5)),bins= 50,norm=LogNorm(vmin=1, vmax=50) ,cmap=cmap, rasterized=True)

bias, scatter = measure_bias_scatter(clean_cat_gswlc_match[xmstar].data,gswlc_match["LOGMSTAR"].data)
ax[ax_id].text( xpos,ypos,rf"b = {bias:.2f}, $\sigma$ = {scatter:.2f}",fontsize = fsize)

 # Create a colorbar
cbar = plt.colorbar(im, ax=ax[ax_id], orientation='horizontal', pad=0.05)
cbar.ax.set_position([
    0.295,   # Left position
    0.62,  # Top position
    ax[ax_id].get_position().width * 0.1,  # Width (40% of plot width)
    0.02  # Height (thin bar)
])    

#######
ax_id = 2
ax[ax_id].set_title(r"Fastspecfit",fontsize = title_size )
h, xedges, yedges, im =  ax[ax_id].hist2d(clean_cat_fspec_match[xmstar],fspec_mstar_f,range= ( (6,9.5),(6,9.5)),bins= 50,norm=LogNorm(vmin=1, vmax=1000) ,cmap=cmap, rasterized=True)

bias, scatter = measure_bias_scatter(clean_cat_fspec_match[xmstar].data,fspec_mstar_f)
ax[ax_id].text( xpos,ypos,rf"b = {bias:.2f}, $\sigma$ = {scatter:.2f}",fontsize = fsize)

cbar = plt.colorbar(im, ax=ax[ax_id], orientation='horizontal', pad=0.05)
cbar.ax.set_position([
    0.545,   # Left position
    0.62,  # Top position
    ax[ax_id].get_position().width * 0.1,  # Width (40% of plot width)
    0.02  # Height (thin bar)
])

#######
ax_id = 3
ax[ax_id].set_title(r"gr-based, Mao et. al.+(2024)",fontsize = title_size )
h, xedges, yedges, im =  ax[ax_id].hist2d(clean_cat[xmstar], clean_cat["LOGM_SAGA_VCMB"] ,range= ( (6,9.5),(6,9.5)),bins= 50,norm=LogNorm(vmin=1, vmax=1000) ,cmap=cmap, rasterized=True)

bias,scatter = measure_bias_scatter(clean_cat[xmstar],clean_cat["LOGM_SAGA_VCMB"]) 
ax[ax_id].text( xpos,ypos,rf"b = {bias:.2f}, $\sigma$ = {scatter:.2f}",fontsize = fsize)

 # Create a colorbar
cbar = plt.colorbar(im, ax=ax[ax_id], orientation='horizontal', pad=0.05)
cbar.ax.set_position([
    0.795,   # Left position
    0.62,  # Top position
    ax[ax_id].get_position().width * 0.1,  # Width (40% of plot width)
    0.02  # Height (thin bar)
])

for i,axi in enumerate(ax):
    axi.set_xlim([6,9.25])
    axi.set_ylim([6,9.25])
    axi.plot([6,11],[6,11],color = "k",lw = 1)
    axi.set_xlabel(r"gr-based $\log_{10}(M_{\bigstar})$",size= 16)
    ax[0].set_ylabel(r"$\log_{10}(M_{\bigstar})$",size= 16)
    ax[0].grid(ls = ":",color = "lightgrey",alpha = 0.5)
    
    xgrid = np.linspace(6,9.25,100)
    make_alternating_plot(ax[i],xgrid,xgrid,dash_len=1,color_1="yellowgreen",color_2="k",lw=1)

    if i != 0:
        axi.set_yticklabels([])

plt.savefig("/global/homes/v/virajvm/DESI2_LOWZ/quenched_fracs_nbs/paper_plots/stellar_mass_comp.pdf",bbox_inches="tight")
plt.show()

